# Semana 4 — Aprendizaje no supervisado: k-means y PCA
### Cuaderno guiado (ALUMNO)

Hasta ahora todos los modelos aprendían de ejemplos con respuesta: la salida de cada fila era conocida
y el modelo aprendía a reproducirla. Eso es aprendizaje **supervisado**. En el aprendizaje
**no supervisado** no hay respuesta: el algoritmo solo ve los datos y busca estructura en ellos.

Este cuaderno es breve y tiene tres partes:

1. **k-means**: agrupar puntos parecidos (*clustering*).
2. **¿Cuántos grupos?**: el método del codo y la silueta.
3. **PCA**: resumir varias variables en dos para poder dibujarlas.

**Cómo trabajar**

- Ejecuta las celdas **en orden**, de arriba abajo.
- En las celdas con `# TODO` hay huecos marcados con `None`. Complétalos siguiendo la **pista** del
  comentario: es una sola línea.
- Después de cada paso hay una celda **Comprueba**. Si da error, revisa ese paso antes de seguir.

## 0. Preparación

Cargamos las librerías. Ejecuta la celda sin cambiar nada.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.decomposition import PCA
from sklearn.datasets import load_iris

## 1. k-means sobre un ejemplo en 2D

Usamos un conjunto de 1000 puntos con dos coordenadas, `x1` y `x2`, y una columna `clase` (de 1 a 4).
k-means **no ve** la columna `clase`: solo la usamos al final, para comprobar si los grupos que
encuentra coinciden con ella.

In [ ]:
datos_ejemplo = pd.read_csv('../datasets/simpleclass_dataset.txt')
print("Filas:", datos_ejemplo.shape[0], "| columnas:", list(datos_ejemplo.columns))
datos_ejemplo.head()

Así ve los datos k-means: puntos sin etiqueta.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(datos_ejemplo['x1'], datos_ejemplo['x2'], s=8, color='gray')
ax.set_xlabel('x1')
ax.set_ylabel('x2')
ax.set_title('Los datos sin etiquetas')
plt.tight_layout()
plt.show()

### Cómo funciona k-means

Se le dice cuántos grupos buscar (*k*) y repite cuatro pasos:

1. Coloca *k* centros al azar.
2. Asigna cada punto al centro más cercano.
3. Mueve cada centro a la media de los puntos que tiene asignados.
4. Repite los pasos 2 y 3 hasta que los centros dejan de moverse.

La cercanía se mide con la distancia euclídea, así que antes se **estandarizan** las variables, como en
la Semana 2: si una variable tuviera valores mucho mayores que otra, dominaría la distancia. El
resultado depende de dónde caigan los centros al principio; por eso `KMeans` lo repite varias veces
(`n_init=10`) y se queda con el mejor, y `random_state=0` hace que salga siempre igual.

Variables de esta sección:

| Variable | Qué contiene |
|---|---|
| `X_ejemplo` | coordenadas `x1` y `x2` estandarizadas |
| `modelo_kmeans` | el k-means con 4 grupos |
| `grupo` | grupo asignado a cada punto (0, 1, 2 o 3) |
| `centros` | coordenadas de los 4 centros finales |

In [ ]:
escalador_ejemplo = StandardScaler()
X_ejemplo = escalador_ejemplo.fit_transform(datos_ejemplo[['x1', 'x2']])

modelo_kmeans = KMeans(n_clusters=4, n_init=10, random_state=0)

# === TODO ===
# Ajusta k-means con X_ejemplo y obtén el grupo de cada punto (una línea).
# pista: grupo = modelo_kmeans.fit_predict(X_ejemplo)
grupo = None   # TODO

centros = modelo_kmeans.cluster_centers_
puntos_por_grupo = np.bincount(grupo)
print("Puntos por grupo:", puntos_por_grupo)

**Comprueba:**

In [ ]:
assert grupo is not None, "Completa el TODO: grupo = modelo_kmeans.fit_predict(X_ejemplo)"
assert len(np.unique(grupo)) == 4, "Deberías tener 4 grupos"
assert sorted(puntos_por_grupo) == [233, 243, 247, 277], "Los grupos deberían tener 233, 243, 247 y 277 puntos"
print("Correcto")

Los grupos encontrados (colores) y sus centros (cruces negras):

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(X_ejemplo[:, 0], X_ejemplo[:, 1], c=grupo, s=8, cmap='tab10')
ax.scatter(centros[:, 0], centros[:, 1], marker='X', s=200, color='black')
ax.set_xlabel('x1 (estandarizada)')
ax.set_ylabel('x2 (estandarizada)')
ax.set_title('Grupos de k-means (k = 4)')
plt.tight_layout()
plt.show()

Ahora sí miramos la columna `clase`: cuántos puntos de cada clase real han caído en cada grupo.

In [ ]:
comparacion = pd.crosstab(grupo, datos_ejemplo['clase'], rownames=['grupo'], colnames=['clase real'])
comparacion

**Pregunta 1.1.** ¿Coinciden los grupos que encuentra k-means con las clases reales? ¿Por qué los
números de grupo no son los mismos que los de clase?

**Tu respuesta:** *escribe aquí*

## 2. ¿Cuántos grupos?

En la sección 1 le dijimos a k-means que buscara 4 grupos, pero con datos reales no se sabe cuántos
hay. Antes de medir nada, veamos qué hace k-means si le pedimos otro número de grupos. La celda
siguiente lo ejecuta con *k* de 2 a 7 sobre los mismos puntos y dibuja los grupos (colores) y sus
centros (cruces negras). Ejecútala sin cambiar nada.

| Variable | Qué contiene |
|---|---|
| `valores_k_panel` | números de grupos que dibujamos, de 2 a 7 |
| `ejes` | los seis gráficos del panel, uno por cada *k* |
| `grupo_panel` | grupo asignado a cada punto, para el *k* que se dibuja |
| `centros_panel` | coordenadas de los centros, para el *k* que se dibuja |

In [ ]:
valores_k_panel = [2, 3, 4, 5, 6, 7]
n_paneles = len(valores_k_panel)

fig, ax = plt.subplots(2, 3, figsize=(12, 7.5))
ejes = ax.flatten()

for posicion in range(n_paneles):
    k = valores_k_panel[posicion]
    modelo_panel = KMeans(n_clusters=k, n_init=10, random_state=0)
    grupo_panel = modelo_panel.fit_predict(X_ejemplo)
    centros_panel = modelo_panel.cluster_centers_

    eje = ejes[posicion]
    eje.scatter(X_ejemplo[:, 0], X_ejemplo[:, 1], c=grupo_panel, s=6, cmap='tab10')
    eje.scatter(centros_panel[:, 0], centros_panel[:, 1], marker='X', s=150, color='black')
    eje.set_title(f'k = {k}')

plt.tight_layout()
plt.show()

**Pregunta 2.1.** ¿Qué hace k-means cuando le pedimos menos grupos que nubes hay en los datos? ¿Y
cuando le pedimos más? Fíjate también en dónde quedan los centros con k = 2 y k = 3.

**Tu respuesta:** *escribe aquí*

### El codo y la silueta

Aquí, con dos variables, podemos dibujar los grupos y mirarlos. Con muchas variables eso no es posible
y hace falta medir. Dos herramientas ayudan a elegir *k*:

- **Inercia y método del codo.** La inercia es la suma, para todos los puntos, de la distancia al
  cuadrado a su centro: cuanto más pequeña, más compactos son los grupos. Siempre baja al aumentar *k*
  (con un grupo por punto valdría 0), así que no se busca el mínimo, sino el **codo**: el *k* a partir
  del cual añadir grupos apenas la reduce.
- **Silueta.** Para cada punto se compara la distancia media a los puntos de su grupo (*a*) con la
  distancia media a los del grupo vecino más cercano (*b*): s = (b − a) / max(a, b). Va de −1 a 1; cerca
  de 1, el punto está bien dentro de su grupo y lejos de los demás. La **silueta media** de todos los
  puntos resume el agrupamiento, y se elige el *k* con la silueta media más alta.

Referencia orientativa para la silueta media: por debajo de 0,25, sin estructura clara; de 0,25 a 0,5,
estructura débil; de 0,5 a 0,7, razonable; por encima de 0,7, fuerte.

Variables de esta sección:

| Variable | Qué contiene |
|---|---|
| `valores_k` | números de grupos que probamos, de 2 a 8 |
| `inercias` | inercia de cada *k* |
| `siluetas` | silueta media de cada *k* |

In [ ]:
valores_k = range(2, 9)
inercias = []
siluetas = []

for k in valores_k:
    modelo_k = KMeans(n_clusters=k, n_init=10, random_state=0)
    grupo_k = modelo_k.fit_predict(X_ejemplo)

    # === TODO ===
    # Calcula la inercia y la silueta media de este agrupamiento.
    # pista: modelo_k.inertia_   y   silhouette_score(X_ejemplo, grupo_k)
    inercia_k = None   # TODO
    silueta_k = None   # TODO

    inercias.append(inercia_k)
    siluetas.append(silueta_k)
    print(f"k = {k}: inercia = {inercia_k:7.1f}   silueta = {silueta_k:.3f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(valores_k, inercias, 'o-')
ax[0].set_xlabel('número de grupos (k)')
ax[0].set_ylabel('inercia')
ax[0].set_title('Método del codo')
ax[1].plot(valores_k, siluetas, 'o-', color='C1')
ax[1].set_xlabel('número de grupos (k)')
ax[1].set_ylabel('silueta media')
ax[1].set_title('Silueta')
for eje in ax:
    eje.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Comprueba:**

In [ ]:
posicion_mejor = np.argmax(siluetas)
k_mejor_silueta = valores_k[posicion_mejor]
assert k_mejor_silueta == 4, "La silueta más alta debería estar en k = 4: revisa silhouette_score(X_ejemplo, grupo_k)"
print("Correcto")

**Pregunta 2.2.** ¿Qué número de grupos sugiere el codo? ¿Y la silueta? ¿Cuánto vale la silueta media
en ese *k* y qué dice la referencia?

**Tu respuesta:** *escribe aquí*

### El gráfico de silueta: mirar más allá de la media

La silueta media resume todo el agrupamiento en un solo número. El **gráfico de silueta** dibuja la
silueta s(i) de **cada punto**, como una barra horizontal, agrupada por grupo y ordenada de mayor a
menor; la línea roja marca la media. Permite ver lo que la media esconde: grupos enteros por debajo de
ella, grupos muy finos o muy gruesos, y puntos con silueta negativa (más cerca del grupo vecino que del
suyo). Dibujamos el de k = 4 y, para comparar, el de k = 6. Ejecuta sin cambiar nada.

| Variable | Qué contiene |
|---|---|
| `silueta_por_punto` | silueta s(i) de cada punto, para el k que se dibuja |
| `n_negativos` | número de puntos con silueta negativa |

In [ ]:
def dibujar_silueta(X, k, eje):
    """Gráfico de silueta de un k-means con k grupos sobre X, en el eje indicado."""
    modelo_dibujo = KMeans(n_clusters=k, n_init=10, random_state=0)
    grupo_dibujo = modelo_dibujo.fit_predict(X)
    silueta_por_punto = silhouette_samples(X, grupo_dibujo)
    silueta_media = silueta_por_punto.mean()

    posicion_inferior = 0
    for numero_grupo in range(k):
        valores_grupo = silueta_por_punto[grupo_dibujo == numero_grupo]
        valores_grupo = np.sort(valores_grupo)
        n_grupo = len(valores_grupo)
        posiciones = np.arange(posicion_inferior, posicion_inferior + n_grupo)
        eje.fill_betweenx(posiciones, 0, valores_grupo, alpha=0.8)
        eje.text(-0.08, posicion_inferior + n_grupo / 2, str(numero_grupo))
        posicion_inferior = posicion_inferior + n_grupo + 15

    eje.axvline(silueta_media, color='red', ls='--')
    eje.axvline(0, color='black', lw=0.8)
    eje.set_xlim(-0.2, 1)
    eje.set_yticks([])
    eje.set_xlabel('silueta de cada punto, s(i)')
    eje.set_title(f'Gráfico de silueta (k = {k}); línea roja: media = {silueta_media:.2f}')

    n_negativos = (silueta_por_punto < 0).sum()
    print(f"k = {k}: silueta media = {silueta_media:.3f}; puntos con silueta negativa: {n_negativos}")


fig, ax = plt.subplots(1, 2, figsize=(11, 4.5), sharey=False)
dibujar_silueta(X_ejemplo, 4, ax[0])
dibujar_silueta(X_ejemplo, 6, ax[1])
ax[0].set_ylabel('puntos, agrupados por grupo')
plt.tight_layout()
plt.show()

**Pregunta 2.3.** Compara los dos gráficos de silueta. ¿Qué te dice el de k = 6 que no te dice su
silueta media? ¿Hay puntos con silueta negativa, y qué significa eso?

**Tu respuesta:** *escribe aquí*

## 3. PCA: ver en 2D datos de 4 variables

El conjunto **Iris** tiene 150 flores de tres especies con 4 medidas cada una: largo y ancho del sépalo
y del pétalo. Viene incluido en scikit-learn, así que no hay que descargar nada.

Con 4 variables no podemos dibujar las flores. La **PCA** (análisis de componentes principales) busca
nuevas direcciones, combinaciones de las variables originales, ordenadas según cuánta variación de los
datos recogen: la **primera componente** es la dirección en la que los datos varían más; la segunda, la
siguiente, perpendicular a la primera; y así sucesivamente. Si nos quedamos con las dos primeras,
proyectamos las flores sobre un plano perdiendo lo mínimo. Como k-means, la PCA **no usa la especie**, y
también se estandarizan antes las variables.

Variables de esta sección:

| Variable | Qué contiene |
|---|---|
| `medidas` | las 4 medidas de cada flor (150 filas) |
| `especie` | especie de cada flor (0, 1 o 2); solo para colorear el dibujo |
| `nombres_especie` | nombre de cada especie |
| `X_iris` | medidas estandarizadas |
| `pca` | la PCA con 2 componentes |
| `componentes` | coordenadas de cada flor en el plano de las dos primeras componentes |
| `varianza_explicada` | fracción de la variación total que recoge cada componente |

In [ ]:
iris = load_iris(as_frame=True)
medidas = iris.data
especie = iris.target
nombres_especie = iris.target_names
print("Flores:", medidas.shape[0], "| medidas:", list(medidas.columns))
medidas.head()

In [ ]:
escalador_iris = StandardScaler()
X_iris = escalador_iris.fit_transform(medidas)

pca = PCA(n_components=2)

# === TODO ===
# Ajusta la PCA y proyecta las flores sobre las dos primeras componentes (una línea).
# pista: componentes = pca.fit_transform(X_iris)
componentes = None   # TODO

varianza_explicada = pca.explained_variance_ratio_
varianza_total = varianza_explicada.sum()
print("Fracción de la variación que recoge cada componente:", varianza_explicada.round(3))
print(f"Las dos juntas: {varianza_total:.1%}")

**Comprueba:**

In [ ]:
assert componentes is not None, "Completa el TODO: componentes = pca.fit_transform(X_iris)"
assert componentes.shape == (150, 2), "Deberías tener 150 flores con 2 coordenadas"
assert abs(varianza_total - 0.958) < 0.005, "Las dos componentes deberían recoger alrededor del 95,8 %"
print("Correcto")

Las flores en el plano de las dos primeras componentes, coloreadas por especie:

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))
for codigo in range(3):
    es_de_esta_especie = especie == codigo
    ax.scatter(componentes[es_de_esta_especie, 0], componentes[es_de_esta_especie, 1],
               s=15, label=nombres_especie[codigo])
ax.set_xlabel('primera componente (PC1)')
ax.set_ylabel('segunda componente (PC2)')
ax.set_title('Iris proyectado con PCA')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Pregunta 3.1.** ¿Qué fracción de la variación de las 4 medidas conservan las dos componentes? En el
dibujo, ¿se separan las tres especies? Recuerda que la PCA no ha usado la especie.

**Tu respuesta:** *escribe aquí*

## Resumen

- **k-means** agrupa puntos parecidos sin usar ninguna etiqueta. Hay que estandarizar y decirle cuántos
  grupos buscar.
- El **codo** de la inercia y la **silueta** ayudan a elegir ese número; la silueta dice además si hay
  grupos claros o no.
- La **PCA** resume muchas variables en unas pocas componentes que recogen la mayor parte de la
  variación; con dos, los datos se pueden dibujar.